In [2]:
!pip install deep-sort-realtime

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 4.7 MB/s eta 0:00:02
   --------------------------- ------------ 5.8/8.4 MB 19.8 MB/s eta 0:00:01
   ----------------------------- ---------- 6.3/8.4 MB 16.2 MB/s eta 0:00:01
   ---------------------------------------  8.4/8.4 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 9.5 MB/s  0:00:01


In [12]:
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

In [13]:
# Load YOLO model
model = YOLO(r"..\fine-tunning2\runs\detect\train\weights\best.pt")

In [14]:
# Initialize DeepSORT
tracker = DeepSort(
    max_age=30,
    n_init=3,
    max_iou_distance=0.7,
    max_cosine_distance=0.2
)

In [15]:
# Open video
cap = cv2.VideoCapture(r"..\video1.mp4")

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    r".\tracking_result1.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (frame_width, frame_height)
)

In [ ]:
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # ---------------- YOLO ----------------
    results = model(frame)

    detections = []

    for result in results:

        for box in result.boxes:

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            conf = float(box.conf[0])

            cls = int(box.cls[0])

            w = x2 - x1
            h = y2 - y1

            detections.append(
                (
                    [x1, y1, w, h],   # bbox
                    conf,             # confidence
                    cls               # class name
                )
            )

    # ---------------- DeepSORT ----------------
    tracks = tracker.update_tracks(
        detections,
        frame=frame
    )

    # ---------------- Draw Tracking ----------------
    for track in tracks:

        if not track.is_confirmed():
            continue

        track_id = track.track_id

        class_id = track.get_det_class()
        class_name = model.names[class_id]

        l, t, r, b = map(int, track.to_ltrb())

        # Bounding box
        cv2.rectangle(
            frame,
            (l, t),
            (r, b),
            (0, 255, 0),
            2
        )

        # Label
        label = f"{class_name} | ID:{track_id}"

        cv2.putText(
            frame,
            label,
            (l, t - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    out.write(frame)

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 Keeper, 16 Players, 1 Ref, 75.7ms
Speed: 2.8ms preprocess, 75.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 2 Refs, 141.8ms
Speed: 3.2ms preprocess, 141.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 85.9ms
Speed: 5.7ms preprocess, 85.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 20 Players, 97.4ms
Speed: 2.6ms preprocess, 97.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 81.8ms
Speed: 3.1ms preprocess, 81.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 20 Players, 1 Ref, 114.2ms
Speed: 2.5ms preprocess, 114.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 21 Players, 135.8ms
Speed: 3.1ms preprocess, 135.8ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 38